### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from pathlib import Path

work_dir = os.getenv("WORK_DIR")

In [2]:
### Read all the documents inside the directory
from com.example.agentic.loader.LoadManager import LoadManager

load_manager = LoadManager(f"{work_dir}/docs")
dir_douments = load_manager.from_directory()

print(f"[*INFO] Total loaded documents: {len(dir_douments)}")


USER_AGENT environment variable not set, consider setting it to identify your requests.


[DEBUG] Data path: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs
[DEBUG] Found 1 PDF files: ['/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf']
[DEBUG] Loading PDF: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf
[DEBUG] Loaded 2 PDF docs from /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf
[DEBUG] Found 3 TXT files: ['/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/machine_learning.txt', '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/python_intro.txt', '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/dharmendra.txt']
[DEBUG] Loading TXT: /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/machine_learning.txt
[DEBUG] Loaded 1 TXT docs from /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/machine_learning.txt
[DEBUG] Loading TXT: /home/brijeshdhaker/IdeaProje

In [3]:
dir_douments

[Document(metadata={'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-04-07T14:01:28+05:30', 'author': 'Brijesh Dhaker', 'moddate': '2026-04-07T14:01:28+05:30', 'source': '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Brijesh K. Dhaker \nMobile: +91 9820937445 \nbrijeshdhaker@gmail.com \nhttps://www.linkedin.com/in/brijeshdhaker \n \n \n \nWORK EXPERIENCE \nDIRECTOR | UBS BUSINESS SOLUTIONS, INDIA | APR-2022 - PRESENT \nGCORC: Financial Crime Prevention \n• AI & Innovation Lead, Client Risk Rating & Compliance Reporting: Formulate \nand execute AI Strategy & Modernization for a brown-field large Client Risk \nRating processing stack into green-field efforts a modern, high-volume, low-\nlatency, and scalable AI & RAG based technology platform. \n• Architecture & Design: Own the architecture and design of CRR & SVC \nenterprise app

In [ ]:
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,length_function=len,separators=["\n\n", "\n", " ", ""])

_chunks = splitter.split_documents(dir_douments)
print(f"[INFO] Split {len(dir_douments)} documents into {len(_chunks)} chunks.")
# Show example of a chunk
if _chunks:
    print(f"Sample Chunk [0]:")
    print(f"Content: {_chunks[0].page_content[:200]}...")
    print(f"Metadata: {_chunks[0].metadata}")


[INFO] Split 5 documents into 12 chunks.
Sample Chunk [0]:
Content: Brijesh K. Dhaker 
Mobile: +91 9820937445 
brijeshdhaker@gmail.com 
https://www.linkedin.com/in/brijeshdhaker 
 
 
 
WORK EXPERIENCE 
DIRECTOR | UBS BUSINESS SOLUTIONS, INDIA | APR-2022 - PRESENT 
GCO...
Metadata: {'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-04-07T14:01:28+05:30', 'author': 'Brijesh Dhaker', 'moddate': '2026-04-07T14:01:28+05:30', 'source': '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
[*INFO] Total splitted chunks: 12


### embedding And vectorStoreDB

In [5]:
### collect the text from chunks
_texts=[doc.page_content for doc in _chunks]

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(_texts)

Loading embedding model: all-MiniLM-L6-v2
Model all-MiniLM-L6-v2 loaded successfully.
Embedding dimension: 384
Generating embeddings for 12 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (12, 384)


### VectorStore

In [7]:
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

vectorstore = VectorStoreManager()



Vector store initialized. Collection: sandbox_documents
Existing documents in collection: 0


In [8]:
## store embeddings into the vector database
#vectorstore.add_documents(_chunks, embeddings)

Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 12


### Retriever Pipeline From VectorStore

In [9]:
from com.example.agentic.vectors.ContextRetriever import ContextRetriever
#
ctx_retriever= ContextRetriever(vectorstore,embedding_manager)
ctx_retriever


In [10]:
ctx_retriever.retrieve("Tell me about Dharmendra ?")

Retrieving documents for query: 'Tell me about Dharmendra ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_a3d62a13_11',
  'content': "Beginning in the late 1990s, he appeared in character roles in several successful and acclaimed films, such as Life in a... Metro, Apne, Johnny Gaddaar, Yamla Pagla Deewana, Rocky Aur Rani Kii Prem Kahaani and Teri Baaton Mein Aisa Uljha Jiya.[11][12][13] In 1997, he received the Filmfare Lifetime Achievement Award for his contributions to Bollywood. He was a member of the 15th Lok Sabha of India, representing the Bikaner constituency in Rajasthan from the Bharatiya Janata Party (BJP).[14] The patriarch of the Deol family, Dharmendra's private life received much attention, particularly his marriages to Prakash Kaur and actress Hema Malini.[15] For his contributions to the arts, the Government of India honoured him with the Padma Bhushan in 2012 and was later posthumously awarded Padma Vibhushan in 2026.",
  'metadata': {'source': '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/dharmendra.txt',
   'doc_index': 11,
   'content_leng

In [11]:
ctx_retriever.retrieve("Tell me about Indian actor ?")


Retrieving documents for query: 'Tell me about Indian actor ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_1d213f86_9',
  'content': 'Dharmendra[a] (8 December 1935 – 24 November 2025) was an Indian actor, producer and politician, primarily known for his work in Hindi films. He is regarded as one of the greatest and most commercially successful actors in the history of Indian cinema.[4] In a career spanning 65 years, he worked in over 300 films, holding the record for starring in the highest number of hit films in Hindi cinema.[3]',
  'metadata': {'doc_index': 9,
   'source': '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/text/dharmendra.txt',
   'content_length': 402},
  'similarity_score': 0.4085502028465271,
  'distance': 0.5914497971534729,
  'rank': 1},
 {'id': 'doc_a3d62a13_11',
  'content': "Beginning in the late 1990s, he appeared in character roles in several successful and acclaimed films, such as Life in a... Metro, Apne, Johnny Gaddaar, Yamla Pagla Deewana, Rocky Aur Rani Kii Prem Kahaani and Teri Baaton Mein Aisa Uljha Jiya.[11][12][13] In 1997, he rece

### RAG Pipeline- VectorDB To LLM Output Generation

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, SystemMessage

In [14]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
            Context: 
            {context}
            
            Question: {question}
            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        Args:
            query: User question
            context: Retrieved context
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}
            Question: {query}
            
            Answer: """
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [15]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [17]:
### get the context from the retriever and pass it to the LLM
ctx = ctx_retriever.retrieve("Tell me about Indian actor ?")
response = groq_llm.generate_response("Tell me about Indian actor ?",context=ctx)
print(response)

Retrieving documents for query: 'Tell me about Indian actor ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
**Dharmendra** (8 December 1935 – 24 November 2025) was a legendary Indian actor, producer and politician.  
- **Film career**: Over a 65‑year span he appeared in more than 300 Hindi films, holding the record for the highest number of hit films in Bollywood. He began with *Dil Bhi Tera Hum Bhi Tere* (1960) and rose to stardom in the 1960s‑80s with classics such as *Sholay*, *Yaadon Ki Baaraat*, *Seeta Aur Geeta*, *Dost*, *Shikar*, *Jugnu*, *Mera Gaon Mera Desh*, and many others. He was often called India’s “He‑Man” for his action roles.  
- **Later work**: From the late 1990s he took on character roles in films like *Life in a Metro*, *Apne*, *Johnny Gaddaar*, *Yamla Pagla Deewana*, *Rocky Aur Rani Kii Prem Kahaani*, and *Teri Baaton Mein Aisa Uljha Jiya*.  
- **Awards & honors**: He received the Filmfare Lifetime Achievement Award (1997), the Padma Bhushan (2012), and was posthumously awarded the Padma V

### Integration Vectordb Context pipeline With LLM output

In [18]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [19]:
answer=rag_simple("What is Dharmendra's profile summary ?",ctx_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Dharmendra's profile summary ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
**Dharmendra** – Indian film actor, producer, and politician.  
- **Career**: Starred in numerous Hindi films from the 1960s onward, with notable roles in *Life in a... Metro*, *Apne*, *Johnny Gaddaar*, *Yamla Pagla Deewana*, *Rocky Aur Rani Kii Prem Kahaani*, and *Teri Baaton Mein Aisa Uljha Jiya*.  
- **Awards**: Filmfare Lifetime Achievement Award (1997); Padma Bhushan (2012); posthumously Padma Vibhushan (2026).  
- **Politics**: Served as a Member of Parliament in the 15th Lok Sabha (Bikaner, Rajasthan) representing the Bharatiya Janata Party.  
- **Personal**: Patriarch of the Deol family; married to Prakash Kaur and actress Hema Malini.


### Enhanced RAG Pipeline Features

In [24]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Dharmendra profile summary ?", ctx_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Dharmendra profile summary ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(ctx_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)

Final Answer: No relevant context found.
Summary: The response indicates that no relevant context was found. Consequently, no additional information can be provided.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The response indicates that no relevant context was found. Consequently, no additional information can be provided.'}
